In [1]:
import os
from elasticsearch import Elasticsearch
from rag_helper import RAGBase, INSTRUCTIONS
from ingest import load_labor_code_data
from google import genai
from dotenv import load_dotenv


# 1. Load the API key from your .env file
load_dotenv()
# 1. Connect to the local Docker container
es_client = Elasticsearch('http://localhost:9200')
index_name = "juris-maroc"

# 2. Define the Elasticsearch mapping (schema)
index_settings = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "content": {"type": "text"},
            "article_number": {"type": "keyword"}
        }
    }
}

# 3. Create the index and push your documents into it
es_client.indices.create(index=index_name, body=index_settings, ignore=[400])
documents = load_labor_code_data()

for doc in documents:
    es_client.index(index=index_name, document=doc)
print(f"Indexed {len(documents)} documents into Elasticsearch.")

# 4. Create the new class inheriting from RAGBase
class ElasticRAG(RAGBase):
    def __init__(self, es_client, index_name, llm_client, instructions):
        # Initialize the parent class without minsearch
        super().__init__(index=None, llm_client=llm_client, instructions=instructions)
        self.es_client = es_client
        self.index_name = index_name

    def search(self, query, num_results=5):
        # Elasticsearch specific query syntax
        search_query = {
            "size": num_results,
            "query": {
                "bool": {
                    "must": {
                        "multi_match": {
                            "query": query,
                            "fields": ["content^3", "article_number"], # Boosts content importance
                            "type": "best_fields"
                        }
                    }
                }
            }
        }

        response = self.es_client.search(index=self.index_name, body=search_query)

        # Extract the source documents from the ES JSON response
        results = [hit['_source'] for hit in response['hits']['hits']]
        return results

# 5. Test the new enterprise pipeline
client = genai.Client()
es_assistant = ElasticRAG(es_client=es_client, index_name=index_name, llm_client=client, instructions=INSTRUCTIONS)

answer = es_assistant.rag("Quels sont les droits d'une femme enceinte ?")
print(answer)

/tmp/ipykernel_6453/518760338.py:27: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.create(index=index_name, body=index_settings, ignore=[400])
/tmp/ipykernel_6453/518760338.py:27: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body=index_settings, ignore=[400])


Indexed 587 documents into Elasticsearch.


/tmp/ipykernel_6453/518760338.py:59: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = self.es_client.search(index=self.index_name, body=search_query)
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Selon le contexte fourni, les droits d'une femme enceinte sont les suivants :

* **Examen médical du travail (Article 327) :** Dans les entreprises soumises à l'obligation de disposer d'un service médical du travail, la femme enceinte a le droit (et l'obligation) de faire l'objet d'un examen médical par le médecin du travail, selon une fréquence dont ce dernier reste juge.

* **Protection contre la discrimination (Article 9) :** En tant que salariée, elle bénéficie de l'interdiction de toute discrimination fondée sur le sexe (notamment en matière d'embauchage, de répartition du travail, de formation, de salaire, d’avancement, d’avantages sociaux, de mesures disciplinaires et de licenciement), du droit de conclure un contrat de travail et d'adhérer à un syndicat professionnel.

*(Note : Le contexte fourni ne contient pas les autres articles spécifiques relatifs au congé de maternité ou aux autres protections particulières de la femme enceinte).*
